# Day 5 homework — evaluation (project)

Same **patterns** as `course/Day_05_Evaluation.ipynb`, but:

- **Docs agent** with **`doc_search`** over **Pydantic** chunks (not the DE FAQ agent).
- Optional: also log **machine-learning** FAQ runs for comparison.

Uses **`./logs`** under the project folder — keep out of git.


## Setup

```bash
cd project
uv add pandas tqdm requests python-frontmatter minsearch openai python-dotenv pydantic-ai
```

Ensure `OPENAI_API_KEY` is set (e.g. `ai_hero/.env`).


In [1]:
import asyncio
import json
import os
import random
import secrets
import threading
from datetime import datetime
from pathlib import Path
import io
import zipfile

import pandas as pd
import requests
import frontmatter
from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessagesTypeAdapter
from dotenv import load_dotenv
from minsearch import Index

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY")

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)


def run_agent_sync(agent, user_prompt: str):
    out = {}
    def _go():
        out["r"] = asyncio.run(agent.run(user_prompt))
    t = threading.Thread(target=_go)
    t.start()
    t.join()
    return out["r"]


In [2]:
def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    r = requests.get(url)
    r.raise_for_status()
    out = []
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    for fi in zf.infolist():
        fn = fi.filename
        fl = fn.lower()
        if not (fl.endswith(".md") or fl.endswith(".mdx")):
            continue
        try:
            with zf.open(fi) as f:
                c = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(c)
                d = post.to_dict()
                d["filename"] = fn
                out.append(d)
        except Exception as e:
            print("skip", fn, e)
    zf.close()
    return out


def sliding_window(seq, size, step):
    result = []
    for i in range(0, len(seq), step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= len(seq):
            break
    return result


pydantic_docs = read_repo_data("pydantic", "pydantic")
chunks = []
for doc in pydantic_docs:
    dc = doc.copy()
    body = dc.pop("content", "") or ""
    for ch in sliding_window(body, 2000, 1000):
        ch.update(dc)
        chunks.append(ch)

MAX_CHUNKS = 400
_rows = chunks if MAX_CHUNKS is None else chunks[:MAX_CHUNKS]
doc_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[],
)
doc_index.fit(_rows)


def doc_search(query: str):
    return doc_index.search(query, num_results=5)


instructions = """
You answer questions about the Pydantic library using doc_search.
Cite source filenames from search results when possible.
""".strip()

doc_agent = Agent(
    "openai:gpt-4o-mini",
    name="pydantic_doc_agent",
    instructions=instructions,
    tools=[doc_search],
)
len(_rows)


400

In [3]:
def _instructions_text(agent) -> str:
    parts = getattr(agent, "_instructions", None) or []
    return "\n\n".join(str(p) for p in parts)


def log_entry(agent, messages, source="user"):
    fts = getattr(agent, "_function_toolset", None)
    tool_names = list(fts.tools.keys()) if fts is not None else []
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
    return {
        "agent_name": agent.name,
        "system_prompt": _instructions_text(agent),
        "model": str(getattr(agent, "_model", "")),
        "tools": tool_names,
        "messages": dict_messages,
        "source": source,
    }


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    fp = LOG_DIR / f"{agent.name}_{ts_str}_{rand_hex}.json"
    with fp.open("w", encoding="utf-8") as f:
        json.dump(entry, f, indent=2, default=serializer)
    return fp


In [4]:
r = run_agent_sync(doc_agent, "How do I use Field with a default factory?")
print(r.output)
log_interaction_to_file(doc_agent, r.new_messages(), source="user")


To use the `Field` function with a default factory in Pydantic, you can provide a callable (like a function) to generate a default value for a field. This is particularly useful for mutable types (like lists or dictionaries) to avoid issues with shared mutable defaults.

Here's a simple example:

```python
from pydantic import BaseModel, Field
from typing import List

def default_list() -> List[int]:
    return [1, 2, 3]

class MyModel(BaseModel):
    my_field: List[int] = Field(default_factory=default_list)

# Example usage
model_instance = MyModel()
print(model_instance.my_field)  # Output: [1, 2, 3]
```

In this example:
- `default_list` is a function that returns a list. It is passed to `Field` as `default_factory`.
- Whenever an instance of `MyModel` is created without providing `my_field`, the list `[1, 2, 3]` will be automatically generated as the default.

### Notes:
1. If you define a mutable default (like a list or dict) directly in the class definition, all instances share t

WindowsPath('logs/pydantic_doc_agent_20260329_000819_5feff7.json')

In [5]:
evaluation_prompt = """
Evaluate an agent answer (<ANSWER>) for question (<QUESTION>) using <INSTRUCTIONS> and <LOG>.

Checklist (true/false + justification):
- instructions_follow, instructions_avoid, answer_relevant, answer_clear
- answer_citations, completeness, tool_call_search
""".strip()


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


eval_agent = Agent(
    "openai:gpt-4o-mini",
    name="eval_agent",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)

fmt = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


def load_log_file(path):
    path = Path(path)
    d = json.loads(path.read_text(encoding="utf-8"))
    d["log_file"] = str(path)
    return d


def extract_qa(messages):
    q = a = None
    for m in messages:
        if m.get("kind") == "request":
            for p in m.get("parts", []):
                if p.get("part_kind") == "user-prompt":
                    c = p.get("content")
                    q = c if isinstance(c, str) else str(c)
    for m in reversed(messages):
        if m.get("kind") == "response":
            for p in m.get("parts", []):
                if p.get("part_kind") == "text":
                    c = p.get("content")
                    a = c if isinstance(c, str) else str(c)
                    break
        if a:
            break
    return q, a


def evaluate_one(log_record):
    msgs = log_record["messages"]
    q, a = extract_qa(msgs)
    up = fmt.format(
        instructions=log_record["system_prompt"],
        question=q,
        answer=a,
        log=json.dumps(msgs),
    )
    return run_agent_sync(eval_agent, up).output


In [6]:
# Evaluate latest log for this agent
logs = sorted(LOG_DIR.glob("pydantic_doc_agent_*.json"))
if logs:
    ev = evaluate_one(load_log_file(logs[-1]))
    print(ev.summary)
    for c in ev.checklist:
        print(c)
else:
    print("No logs yet.")


The response provides a thorough explanation of how to use Field with a default factory in Pydantic, but lacks citation of specific source documentation.
check_name='instructions_follow' justification='The answer adheres to the requirement to explain usage of the Field with a default factory in Pydantic.' check_pass=True
check_name='instructions_avoid' justification='The answer does not contain any irrelevant or disallowed content.' check_pass=True
check_name='answer_relevant' justification='The answer directly addresses how to use Field with a default factory in Pydantic, providing an example and explanations related to the topic.' check_pass=True
check_name='answer_clear' justification='The explanation is clear, along with an example, which helps in understanding the concept.' check_pass=True
check_name='answer_citations' justification='The answer references the Pydantic documentation, fitting guidance for further information, but does not cite specific source filenames; it should ha

## Batch evaluation & Pandas

After you have **several** logs (including `source='ai-generated'`), aggregate pass rates per checklist item.

In [9]:
eval_set = []
for log_file in LOG_DIR.glob("pydantic_doc_agent_*.json"):
    rec = load_log_file(log_file)
    if rec.get("source") == "ai-generated":
        eval_set.append(rec)

eval_results = []
for rec in tqdm(eval_set):
    eval_results.append((rec, evaluate_one(rec)))

rows = []
for rec, ev in eval_results:
    q, a = extract_qa(rec["messages"])
    row = {"file": Path(rec["log_file"]).name, "question": q, "answer": (a or "")[:500]}
    for c in ev.checklist:
        row[c.check_name] = c.check_pass
    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    print(df.head())
    print(df.mean(numeric_only=True))
else:
    print("No ai-generated logs yet — run the question generator cell below first.")

  0%|          | 0/5 [00:00<?, ?it/s]

                                             file  \
0  pydantic_doc_agent_20260329_000843_2b9156.json   
1  pydantic_doc_agent_20260329_000853_79020e.json   
2  pydantic_doc_agent_20260329_000900_2f16a9.json   
3  pydantic_doc_agent_20260329_000905_9a89b5.json   
4  pydantic_doc_agent_20260329_000910_b9a68d.json   

                                            question  \
0  What is the purpose of the `create_model` meth...   
1  How can I validate the assignments to a model ...   
2  What are the performance improvements in Pydan...   
3  What changes do I need to consider when migrat...   
4  How does Pydantic handle serialization for `da...   

                                              answer  instructions_follow  \
0  The `create_model` method in Pydantic is used ...                 True   
1  In Pydantic, you can validate assignments to m...                 True   
2  The performance improvements in Pydantic v2.5....                 True   
3  When migrating from Pydantic v0.7

## AI-generated questions

Sample chunks from the index → LLM writes natural questions → run **`doc_agent`** → log with `source='ai-generated'`. Repeat until you have **≥10** logs for homework.

In [8]:
qg_prompt = """
You help create test questions for an assistant that answers about the Pydantic Python library.

From each chunk of documentation text, output one realistic question a developer might ask.
Questions should vary: API usage, validation, serialization, performance, migration tips.
""".strip()


class QuestionsList(BaseModel):
    questions: list[str]


question_generator = Agent(
    "openai:gpt-4o-mini",
    name="question_generator",
    instructions=qg_prompt,
    output_type=QuestionsList,
)

sample_rows = random.sample(_rows, min(10, len(_rows)))
snippet_docs = [r.get("chunk", "")[:1500] for r in sample_rows]
qg = run_agent_sync(question_generator, json.dumps(snippet_docs))
questions = qg.output.questions

for q in tqdm(questions):
    print(q)
    res = run_agent_sync(doc_agent, q)
    print(res.output[:400], "...\n")
    log_interaction_to_file(doc_agent, res.new_messages(), source="ai-generated")

  0%|          | 0/5 [00:00<?, ?it/s]

What is the purpose of the `create_model` method in Pydantic?
The `create_model` method in Pydantic is used to programmatically create new Pydantic models. This method allows you to define a model dynamically, which can be particularly useful in scenarios where the model's structure is not known until runtime. 

You can specify the fields and their types directly when calling `create_model`, which enables flexibility in model creation. For example, you might ...

How can I validate the assignments to a model field using Pydantic?
In Pydantic, you can validate assignments to model fields by using the `validate_assignment` configuration in your model's configuration class. This feature enables validation of values when fields are updated or set via assignment.

Here's how to set it up:

1. **Enable `validate_assignment`**: You need to create a nested `Config` class inside your model. Set `validate_assignment = True` within t ...

What are the performance improvements in Pydantic v2.5.0b1

## Homework

- Collect **≥10** interaction logs (mix of manual and AI-generated is fine).
- Run the judge on your logs; inspect failures and iterate on **instructions**.
- Duplicate `doc_agent` with a second `instructions` string (e.g. stricter citation rules) and re-run the **same** questions; compare `df.mean(numeric_only=True)` between runs.
- Optional: use a different judge model string in `eval_agent` (e.g. `openai:gpt-4o`) for a second opinion.

[Course](https://alexeygrigorev.com/aihero/)
